In [ ]:
# CELL: Install dependencies
# (Do not touch NumPy — Colab ships a system-managed version already.)
!pip install -q groq scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.1 MB/s eta 0:00:00


In [5]:
# CELL: Imports
import json
import random
import traceback
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from sklearn.ensemble import IsolationForest

In [6]:
# CELL: Load Groq API key from Colab Secrets (never hard-code the key)
from google.colab import userdata
from groq import Groq

API_KEY_NAME = "luffy"
GROQ_MODEL = "openai/gpt-oss-120b"

GROQ_CLIENT = None
GROQ_ERROR = None

try:
    GROQ_API_KEY = userdata.get(API_KEY_NAME)
    if not GROQ_API_KEY:
        GROQ_ERROR = (
            f"Colab Secret '{API_KEY_NAME}' is empty or was not granted notebook access. "
            "Open the key icon in the left sidebar, add a secret named 'luffy' with your "
            "Groq API key, and toggle 'Notebook access' on."
        )
    else:
        GROQ_CLIENT = Groq(api_key=GROQ_API_KEY)
except Exception as e:
    GROQ_ERROR = (
        f"Could not read Colab Secret '{API_KEY_NAME}': {e}. "
        "Make sure you added it in the 🔑 Secrets tab and enabled notebook access."
    )

if GROQ_ERROR:
    print("⚠️ ", GROQ_ERROR)
else:
    print("✅ Groq client initialized. Model:", GROQ_MODEL)

✅ Groq client initialized. Model: openai/gpt-oss-120b


In [7]:
# CELL: SYNTHETIC DATA — simulated machines + historical sensor readings
# ----------------------------------------------------------------------------
# This entire cell is synthetic demo data generation. Nothing here is a real
# industrial reading, and nothing here encodes a diagnosis rule -- it only
# creates plausible sensor histories for the ML/LLM stages below to analyze.

MACHINES = {
    "Pump-01":        {"temperature": 65, "vibration": 2.0, "pressure": 7.5, "operating_hours": 4200},
    "Motor-02":       {"temperature": 70, "vibration": 2.5, "pressure": 7.0, "operating_hours": 5800},
    "Compressor-03":  {"temperature": 72, "vibration": 2.7, "pressure": 8.0, "operating_hours": 7100},
    "Fan-04":         {"temperature": 60, "vibration": 1.5, "pressure": 6.5, "operating_hours": 2900},
}

N_HISTORY = 100

def generate_history(machine_name, baseline, n=N_HISTORY, seed=None):
    """Generate n historical readings with realistic random noise. Only
    Compressor-03 gets a simulated degradation trend in its later readings,
    purely as demo data -- the LLM is never told 'this machine is degrading'."""
    rng = np.random.default_rng(seed)
    temps, vibs, press = [], [], []

    degrade = (machine_name == "Compressor-03")
    degrade_start = int(n * 0.6)

    for i in range(n):
        t = baseline["temperature"] + rng.normal(0, 0.8)
        v = baseline["vibration"] + rng.normal(0, 0.15)
        p = baseline["pressure"] + rng.normal(0, 0.25)

        if degrade and i >= degrade_start:
            progress = (i - degrade_start) / max(1, (n - degrade_start))
            t += progress * 9.0
            v += progress * 1.3
            p -= progress * 1.6

        temps.append(round(float(t), 2))
        vibs.append(round(float(v), 2))
        press.append(round(float(p), 2))

    now = datetime.now()
    timestamps = [now - timedelta(hours=(n - i)) for i in range(n)]

    return pd.DataFrame({
        "timestamp": timestamps,
        "temperature": temps,
        "vibration": vibs,
        "pressure": press,
    })

HISTORY = {
    name: generate_history(name, baseline, seed=idx)
    for idx, (name, baseline) in enumerate(MACHINES.items())
}

print("Generated", N_HISTORY, "synthetic historical readings for each of", len(MACHINES), "machines.")

Generated 100 synthetic historical readings for each of 4 machines.


In [8]:
# CELL: MACHINE LEARNING — Isolation Forest statistical anomaly detection
# ----------------------------------------------------------------------------
# This is the ONLY machine-learning stage. It flags statistical outliers in
# the sensor feature space. It does NOT diagnose faults and is NOT a failure
# predictor -- it is supporting evidence handed to the LLM.

def run_anomaly_detection(df):
    features = df[["temperature", "vibration", "pressure"]].values
    model = IsolationForest(n_estimators=150, contamination=0.08, random_state=42)
    preds = model.fit_predict(features)          # -1 = anomaly, 1 = normal
    scores = model.decision_function(features)   # lower = more anomalous

    out = df.copy()
    out["is_anomaly"] = preds == -1
    out["anomaly_score"] = scores
    return out


def compute_trend_evidence(df, live_values):
    """Descriptive statistics only -- no thresholds, no diagnosis. This is
    packaged as evidence for the LLM to reason over."""
    recent = df.tail(15)
    evidence = {}

    for sensor in ["temperature", "vibration", "pressure"]:
        values = recent[sensor].values
        if values[-1] > values[0]:
            direction = "increasing"
        elif values[-1] < values[0]:
            direction = "decreasing"
        else:
            direction = "stable"
        rate = (values[-1] - values[0]) / max(1, len(values))

        evidence[sensor] = {
            "recent_values": [round(float(v), 2) for v in values[-8:]],
            "direction": direction,
            "rate_of_change_per_reading": round(float(rate), 4),
            "live_value": round(float(live_values[sensor]), 2),
            "historical_average": round(float(df[sensor].mean()), 2),
        }

    evidence["sensor_relationships"] = {
        "temperature_vibration_correlation": round(float(np.corrcoef(df["temperature"], df["vibration"])[0, 1]), 3),
        "temperature_pressure_correlation": round(float(np.corrcoef(df["temperature"], df["pressure"])[0, 1]), 3),
        "vibration_pressure_correlation": round(float(np.corrcoef(df["vibration"], df["pressure"])[0, 1]), 3),
    }

    anomalies = df[df["is_anomaly"]]
    evidence["anomalies"] = {
        "count": int(len(anomalies)),
        "percentage_of_history": round(100 * len(anomalies) / len(df), 1),
        "recent_anomalous_readings": (
            anomalies.tail(5)[["temperature", "vibration", "pressure"]].round(2).to_dict("records")
        ),
    }
    return evidence

print("Anomaly detection + trend evidence functions ready.")

Anomaly detection + trend evidence functions ready.


In [9]:
# CELL: LLM REASONING — maintenance knowledge base + system prompt + Groq call
# ----------------------------------------------------------------------------
# Everything in this cell is the LLM-reasoning stage. The LLM -- not a
# threshold rule -- decides risk level, condition, causes, and actions.

MAINTENANCE_KNOWLEDGE = """
TEMPERATURE-related issues can indicate: overheating, friction, cooling-system problems,
excessive load, or lubrication problems.

VIBRATION-related issues can indicate: imbalance, misalignment, bearing problems,
looseness, or general mechanical wear.

PRESSURE-related issues can indicate: flow restrictions, leakage, valve problems,
process changes, or equipment performance changes.

Sensor trends, rates of change, and the RELATIONSHIP between multiple sensors moving
together are far more informative than checking any single sensor against one fixed
threshold. Reason holistically.
"""

SYSTEM_PROMPT = f"""You are FixAI, an industrial predictive-maintenance AI agent.

Do not behave as a simple threshold classifier. Do not use hard-coded rules such as
"temperature > X = failure". Instead, reason using all of the following:
1. current (live) sensor values
2. historical trends
3. direction of change
4. rate of change
5. relationships between sensors
6. operating hours
7. statistical anomalies (Isolation Forest)
8. the maintenance knowledge base provided to you

Isolation Forest anomalies are supporting evidence only -- an anomaly does NOT
automatically mean machine failure. Do not invent measurements that were not given to
you. Clearly distinguish evidence from uncertainty in your explanation.

MAINTENANCE KNOWLEDGE BASE:
{MAINTENANCE_KNOWLEDGE}

You must respond with ONLY a single valid JSON object (no markdown fences, no extra
text) with EXACTLY these keys:
{{
  "risk_percentage": <number 0-100>,
  "risk_level": "LOW" | "MEDIUM" | "HIGH" | "CRITICAL",
  "condition": "<one sentence describing current machine condition>",
  "confidence": <number 0-100>,
  "detected_signals": ["<short signal>", ...],
  "possible_causes": ["<short cause>", ...],
  "recommended_actions": ["<short action>", ...],
  "maintenance_priority": "ROUTINE" | "MONITOR" | "SCHEDULE_INSPECTION" | "HIGH_PRIORITY",
  "agent_decision": "<one sentence: what the autonomous agent decided to do>",
  "explanation": "<2-4 sentence reasoned explanation using the evidence given>"
}}
"""

ASSESSMENT_DEFAULTS = {
    "risk_percentage": 0, "risk_level": "LOW", "condition": "Unknown",
    "confidence": 0, "detected_signals": [], "possible_causes": [],
    "recommended_actions": [], "maintenance_priority": "ROUTINE",
    "agent_decision": "No decision produced.", "explanation": "No explanation produced.",
}


def call_groq_json(system_prompt, user_prompt, max_retries=2):
    if GROQ_CLIENT is None:
        raise RuntimeError(GROQ_ERROR or "Groq client is not initialized.")

    last_err = None
    for _ in range(max_retries + 1):
        try:
            resp = GROQ_CLIENT.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.3,
                max_tokens=1500,
                response_format={"type": "json_object"},
            )
            content = resp.choices[0].message.content
            return json.loads(content)
        except json.JSONDecodeError as e:
            last_err = f"AI returned malformed JSON: {e}"
        except Exception as e:
            last_err = f"Groq API error: {e}"
    raise RuntimeError(last_err)


def get_ai_assessment(machine_name, machine_meta, live_values, trend_evidence):
    user_prompt = f"""
MACHINE: {machine_name}
OPERATING HOURS: {machine_meta['operating_hours']}

LIVE SENSOR READING:
- Temperature: {live_values['temperature']:.2f}
- Vibration: {live_values['vibration']:.2f}
- Pressure: {live_values['pressure']:.2f}

HISTORICAL TREND EVIDENCE:
{json.dumps(trend_evidence, indent=2)}

Based on all of this evidence (and nothing else), produce your reasoned maintenance
assessment as the JSON object described in your system instructions.
"""
    data = call_groq_json(SYSTEM_PROMPT, user_prompt)
    for k, v in ASSESSMENT_DEFAULTS.items():
        data.setdefault(k, v)
    return data


def ask_fixai(question, machine_name, machine_meta, live_values, trend_evidence):
    system = (
        "You are FixAI, an industrial predictive-maintenance assistant. Answer strictly "
        "from the evidence supplied in the user message. Never invent sensor numbers that "
        "were not given to you. Respond with ONLY JSON: {\"answer\": \"...\"}"
    )
    user_prompt = f"""
MACHINE: {machine_name}
OPERATING HOURS: {machine_meta['operating_hours']}
LIVE VALUES: {json.dumps(live_values)}
RECENT TREND EVIDENCE: {json.dumps(trend_evidence, indent=2)}

USER QUESTION: "{question}"

Answer in 2-5 sentences, using only the evidence above.
"""
    data = call_groq_json(system, user_prompt)
    return data.get("answer", "No answer produced.")

print("LLM reasoning functions ready.")

LLM reasoning functions ready.


In [11]:
# CELL: DASHBOARD / UI -- widgets, layout and rendering
# ----------------------------------------------------------------------------
# Everything below is presentation. No diagnosis logic lives here -- this
# cell only displays whatever the ML + LLM stages above produced.

current_machine = widgets.Dropdown(
    options=list(MACHINES.keys()), value="Pump-01", description="Machine:"
)

temp_slider = widgets.FloatSlider(description="Temp (°C)", readout_format=".1f")
vib_slider = widgets.FloatSlider(description="Vibration", readout_format=".2f")
press_slider = widgets.FloatSlider(description="Pressure", readout_format=".2f")

run_button = widgets.Button(description="RUN AI ANALYSIS", button_style="success", icon="play")
reset_button = widgets.Button(description="RESET", button_style="warning", icon="undo")

ask_input = widgets.Text(placeholder="Ask FixAI a question about this machine...",
                          layout=widgets.Layout(width="70%"))
ask_button = widgets.Button(description="Ask FixAI", icon="comment")

output_area = widgets.Output()
chat_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="6px", max_height="220px", overflow="auto"))

LAST_STATE = {}


def slider_ranges(baseline):
    return {
        "temperature": (baseline["temperature"] - 15, baseline["temperature"] + 25),
        "vibration": (max(0.0, baseline["vibration"] - 1.5), baseline["vibration"] + 4),
        "pressure": (max(0.0, baseline["pressure"] - 3), baseline["pressure"] + 3),
    }


def set_sliders_to_baseline(machine_name):
    baseline = MACHINES[machine_name]
    ranges = slider_ranges(baseline)
    temp_slider.min, temp_slider.max = ranges["temperature"]
    vib_slider.min, vib_slider.max = ranges["vibration"]
    press_slider.min, press_slider.max = ranges["pressure"]
    temp_slider.value = baseline["temperature"]
    vib_slider.value = baseline["vibration"]
    press_slider.value = baseline["pressure"]


set_sliders_to_baseline(current_machine.value)


def on_machine_change(change):
    if change["name"] == "value" and change["new"] != change["old"]:
        set_sliders_to_baseline(change["new"])
        with output_area:
            clear_output()


current_machine.observe(on_machine_change, names="value")


def on_reset(_b):
    set_sliders_to_baseline(current_machine.value)
    with output_area:
        clear_output()


reset_button.on_click(on_reset)


def render_dashboard(machine_name, df, live_values, assessment, trend_evidence):
    meta = MACHINES[machine_name]

    display(HTML(f"""
    <div style="font-family:sans-serif">
      <h2 style="margin-bottom:0">⚙️ FIXAI</h2>
      <p style="color:gray;margin-top:0">AI Predictive Maintenance Platform &mdash;
      Machine: <b>{machine_name}</b></p>
    </div>
    """))

    display(HTML(f"""
    <table style="border-collapse:collapse;width:100%;font-family:sans-serif">
      <tr style="background:#f0f0f0">
        <th style="padding:6px;border:1px solid #ccc">Temperature</th>
        <th style="padding:6px;border:1px solid #ccc">Vibration</th>
        <th style="padding:6px;border:1px solid #ccc">Pressure</th>
        <th style="padding:6px;border:1px solid #ccc">Operating Hours</th>
        <th style="padding:6px;border:1px solid #ccc">AI Risk</th>
        <th style="padding:6px;border:1px solid #ccc">AI Confidence</th>
      </tr>
      <tr>
        <td style="padding:6px;border:1px solid #ccc;text-align:center">{live_values['temperature']:.1f} °C</td>
        <td style="padding:6px;border:1px solid #ccc;text-align:center">{live_values['vibration']:.2f} mm/s</td>
        <td style="padding:6px;border:1px solid #ccc;text-align:center">{live_values['pressure']:.2f} bar</td>
        <td style="padding:6px;border:1px solid #ccc;text-align:center">{meta['operating_hours']}</td>
        <td style="padding:6px;border:1px solid #ccc;text-align:center"><b>{assessment['risk_percentage']}%</b> ({assessment['risk_level']})</td>
        <td style="padding:6px;border:1px solid #ccc;text-align:center">{assessment['confidence']}%</td>
      </tr>
    </table>
    """))

    risk_colors = {"LOW": "#2e7d32", "MEDIUM": "#f9a825", "HIGH": "#ef6c00", "CRITICAL": "#c62828"}
    color = risk_colors.get(assessment["risk_level"], "#555")
    display(HTML(f"""
      <h3>AI Risk Assessment</h3>
      <p style="font-size:1.3em"><b style="color:{color}">{assessment['risk_percentage']}%
      &mdash; {assessment['risk_level']}</b> &nbsp;|&nbsp; Confidence: {assessment['confidence']}%</p>
      <h3>Machine Condition</h3>
      <p>{assessment['condition']}</p>
      <h3>AI Reasoning</h3>
      <p>{assessment['explanation']}</p>
    """))

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    labels = {"temperature": "Temperature (°C)", "vibration": "Vibration (mm/s)", "pressure": "Pressure (bar)"}
    for ax, sensor in zip(axes, ["temperature", "vibration", "pressure"]):
        ax.plot(df["timestamp"], df[sensor], color="#1565c0", linewidth=1)
        anomalies = df[df["is_anomaly"]]
        ax.scatter(anomalies["timestamp"], anomalies[sensor], color="red", s=18, label="anomaly", zorder=5)
        ax.set_title(labels[sensor])
        ax.tick_params(axis="x", rotation=30, labelsize=7)
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    def list_html(title, items):
        lis = "".join(f"<li>{item}</li>" for item in items) or "<li>None reported</li>"
        return f"<h3>{title}</h3><ul>{lis}</ul>"

    display(HTML(list_html("Detected Signals", assessment["detected_signals"])))
    display(HTML(list_html("Possible Causes", assessment["possible_causes"])))
    display(HTML(list_html("Recommended Actions", assessment["recommended_actions"])))

    anomaly_info = trend_evidence["anomalies"]
    display(HTML(f"""
      <h3>Statistical Anomalies
        <span style="font-size:0.6em;color:gray">(Isolation Forest &mdash; statistical outlier
        detection, NOT a failure prediction)</span></h3>
      <p>Anomalies detected: <b>{anomaly_info['count']}</b>
      ({anomaly_info['percentage_of_history']}% of history)</p>
    """))
    if anomaly_info["recent_anomalous_readings"]:
        display(pd.DataFrame(anomaly_info["recent_anomalous_readings"]))

    display(HTML(f"""
      <h3>🤖 Autonomous Maintenance Agent</h3>
      <p><b>Agent Decision:</b> {assessment['agent_decision']}</p>
      <p><b>Maintenance Priority:</b> {assessment['maintenance_priority']}</p>
    """))


def on_run_analysis(_b):
    machine_name = current_machine.value
    live_values = {
        "temperature": temp_slider.value,
        "vibration": vib_slider.value,
        "pressure": press_slider.value,
    }
    with output_area:
        clear_output(wait=True)
        print("Running Isolation Forest + AI reasoning... please wait.")
        try:
            if GROQ_CLIENT is None:
                raise RuntimeError(GROQ_ERROR or "Groq client not initialized.")

            base_df = HISTORY[machine_name]
            live_row = pd.DataFrame([{
                "timestamp": datetime.now(),
                "temperature": live_values["temperature"],
                "vibration": live_values["vibration"],
                "pressure": live_values["pressure"],
            }])
            combined_df = pd.concat([base_df, live_row], ignore_index=True)

            df_scored = run_anomaly_detection(combined_df)
            trend_evidence = compute_trend_evidence(df_scored, live_values)
            assessment = get_ai_assessment(machine_name, MACHINES[machine_name], live_values, trend_evidence)

            LAST_STATE.clear()
            LAST_STATE.update({
                "machine_name": machine_name,
                "trend_evidence": trend_evidence,
                "live_values": live_values,
            })

            clear_output(wait=True)
            render_dashboard(machine_name, df_scored, live_values, assessment, trend_evidence)
        except Exception as e:
            clear_output(wait=True)
            display(HTML(
                f"<div style='color:#c62828;border:1px solid #c62828;padding:10px;"
                f"font-family:sans-serif'><b>⚠️ Error running analysis:</b><br>{e}</div>"
            ))


run_button.on_click(on_run_analysis)


def on_ask(_b):
    question = ask_input.value.strip()
    with chat_output:
        if not question:
            print("Type a question first.")
            return
        if not LAST_STATE:
            print("Run an AI analysis first so FixAI has context to answer from.")
            return
        print("You:", question)
        try:
            answer = ask_fixai(
                question,
                LAST_STATE["machine_name"],
                MACHINES[LAST_STATE["machine_name"]],
                LAST_STATE["live_values"],
                LAST_STATE["trend_evidence"],
            )
            print("FixAI:", answer)
            print("-" * 40)
        except Exception as e:
            print("⚠️ Error:", e)
            print("-" * 40)
    ask_input.value = ""


ask_button.on_click(on_ask)

dashboard = widgets.VBox([
    widgets.HTML("<h1>⚙️ FIXAI</h1><p style=\"color:gray\">AI Predictive Maintenance Platform</p>"),
    current_machine,
    widgets.HTML("<h3>Live Controls</h3>"),
    temp_slider, vib_slider, press_slider,
    widgets.HBox([run_button, reset_button]),
    output_area,
    widgets.HTML("<h3>💬 Ask FixAI</h3>"),
    widgets.HBox([ask_input, ask_button]),
    chat_output,
])

display(dashboard)

You: Why is my system making lot of noise
FixAI: The recent vibration readings, which have been high (up to 3.98) and only now decreasing to 2.7, indicate mechanical agitation that can produce noise. Temperature has been elevated (up to 82.43) and is strongly positively correlated with vibration (0.88), suggesting heat‑induced expansion is amplifying the vibration. Additionally, pressure has spiked to 8.0, deviating from its historical average, which can further stress components and contribute to acoustic emissions. These combined sensor trends explain the increased noise you are hearing.
----------------------------------------
